# Choose Your Own Basin

This tutorial provides an example of how to integrate RAPID2 into a workflow using `Python` to handle model setup and examination of model output. The code below allows for the selection of several model parameters at the start, such as basin, simulation date, and input runoff model, and then performs model setup automatically. Several python packages are used in this tutorial, but all are included in the RAPID2 pypi package except for Matplotlib, which is a commonly used Python plotting package. 

This tutorial is provided as an interactive python notebook (`ipynb`), and can be opend and run in your browser by clicking the Binder badge:

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/c-h-david/rapid-hub/pythonNotebooks?labpath=docs/user-guide/examples/tutorial-anyBasin.ipynb)

Before starting, esnure that RAPID2 has been installed in your vistual environment (see the [Quickstart Guide](https://c-h-david.github.io/rapid-hub/user-guide/quick-start-sandbox/)). If running this tutorial in Binder, this setup is done automatically.

First, import Python packages needed for the tutorial.

In [11]:
import os
import zipfile
from pathlib import Path
import urllib.request
import yaml
import netCDF4
import numpy as np
#import matplotlib.pyplot as plt
import toyplot
from datetime import datetime
import getpass
import earthaccess
import rapid2 as r2

We can check the version of the installed RAPID2 version using:

In [2]:
print(r2.__version__)

2.0.0b5


## 1. Set Parameters
Here is where we set the parameters for the run:
- *BASIN*: Set the basin ID based on global Pfafstetter Level 2 hydrologic regions (e.g. pfaf_74 for the Mississippi Basin)
- *PHASE*: Phase of GLDAS forcing to use (e.g. 2.1)
- *MODEL*: The hydrologic model to use for input (options are: VIC, CLSM, NOAH)
- *TIME*: The year and month for the simulation (e.g.'2010-01')

Note that all input paramaters must be provided as strings.

In [3]:
BASIN = "pfaf_31"
PHASE = '2.1'
MODEL = 'VIC'
TIME = '2010-01'

Next we set up the working directory, and input and output directories.

In [4]:
WORK_DIR = Path("rapid2_tutorial")
INPUT_DIR = WORK_DIR / "input"
OUTPUT_DIR = WORK_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Donwload Static River Parameters

Next, we need to get six static parameter files for the Mississippi Basin. We'll be using prepared files from a Zenodo repository:

[RAPID2 static files for global implementation with MERIT Basins v1.0 and GLDAS v2.0](https://zenodo.org/records/20672740)

This repository includes 61 zipped folders corresponding to the global Pfafstetter Level 2 hydrologic regions. The code below uses the pfaf ID set in the `BASIN` variable above to download and unpack the corresponding zipped folder.

In [5]:
ZENODO_RECORD = "20672740"
ZENODO_BASE = f"https://zenodo.org/records/{ZENODO_RECORD}/files"

print(f"Downloading {BASIN} ZIP file from Zenodo record {ZENODO_RECORD}...")
url = f"{ZENODO_BASE}/MERIT_Basins_v1.0_{BASIN}_GLDAS_v2.0_RAPID_v2.0.zip?download=1"
dest = INPUT_DIR / BASIN 
if not dest.exists():
    urllib.request.urlretrieve(url, dest)
else:
    print(f'\tFiles already downloaded')
with zipfile.ZipFile(dest, "r") as z:
        z.extractall(INPUT_DIR)
print(f"\nStatic files for {BASIN}:")
for f in sorted(INPUT_DIR.glob(f"*{BASIN}*")):
    print(f"  {f.name}")

https://zenodo.org/records/20672740/files/MERIT_Basins_v1.0_pfaf_31_GLDAS_v2.0_RAPID_v2.0.zip?download=1
rapid2_tutorial/input/pfaf_31

Static files for pfaf_31:
  bas_pfaf_31_topo.parquet
  con_pfaf_31.parquet
  cpl_pfaf_31_GLDAS.parquet
  crd_pfaf_31.parquet
  kpr_pfaf_31_nrm.parquet
  pfaf_31
  xpr_pfaf_31_nrm.parquet


## 3. Downlod NLDAS Runoff Data

The RAPID2 routing model requires runoff inputs from a land surface model. We'll use runoff data downloaded from the Global Land Data Assimilation System (GLDAS) using the `dgldas2` command that is part of the `rapid2` package.

A NASA Earthdata account is required to download the GLDAS data. If you have not yet done so, you may also need to accept an end-user license agreement from the NASA Goddard Earth Sciences (GES) Data and Information Services Center (DISC).

> ⚠️ If running this tutorial in Binder, you first need to authenticate with NASA Earthdata using an Earthdata token. Generate one at [urs.earthdata.nasa.gov](https://urs.earthdata.nasa.gov) under "Generate Token", then run the cell below and paste it at the prompt. If running everything in your terminal, the `dgldas2` command will handle your login credentials and the cell below can be skipped.

> ⚠️ Because Binder runs whatever code is in the repository it points at, launch this tutorial only from the badge on the [official rapid-hub repository](https://github.com/c-h-david/rapid-hub).

In [ ]:
# Skip this cell if running in terminal

# Held in an environment variable, which the %%bash cells below inherit so that
# dgldas2 can authenticate.
os.environ["EARTHDATA_TOKEN"] = getpass.getpass("Earthdata token: ").strip()

if not os.environ["EARTHDATA_TOKEN"]:
    raise RuntimeError(
        "[ERROR] No token entered. Generate one at https://urs.earthdata.nasa.gov "
        "under 'Generate Token', then re-run this cell."
    )

auth = earthaccess.login(strategy="environment")

if not auth.authenticated:
    raise RuntimeError("[ERROR] Earthdata login failed - re-run this cell.")

print("[OK] - Earthdata token registered for this session.")

In [7]:
GLDAS_FILE = INPUT_DIR / f'GLDAS_2.1_VIC_{TIME}.nc4'
r2.pull.gldas2(
    YS_phs=PHASE, 
    YS_mod=MODEL, 
    YS_tim=TIME, 
    lsm_ncf=GLDAS_FILE
    )

QUEUEING TASKS | : 100%|██████████| 248/248 [00:00<00:00, 51281.18it/s]
PROCESSING TASKS | : 100%|██████████| 248/248 [00:44<00:00,  5.54it/s]
COLLECTING RESULTS | : 100%|██████████| 248/248 [00:00<00:00, 1541018.36it/s]
Concatenating LSM files: 100%|██████████| 248/248 [00:03<00:00, 68.31it/s]


The new file is placed in `input/Tutorial`. Other files are also automatically
downloaded in that directory and removed after use.

## 4. Coupling the Data with `cpllsm`

Once you have the raw land surface model data and the static river basin parameters, the two must be linked by mapping the runoff to the river network.. The `cpllsm` tool transforms the gridded runoff data into RAPID2's reach-based external inflow format. By providing the land surface model data, connectivity, coordinates, and coupling files, this tool generates the external inflow NetCDF file required for routing.

In [8]:
QEXT_FILE = INPUT_DIR / f"Qext_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"
r2.prep.couple(
    lsm_ncf=GLDAS_FILE, 
    con_pqt=f'{INPUT_DIR}/con_{BASIN}.parquet', 
    crd_pqt=f'{INPUT_DIR}/crd_{BASIN}.parquet', 
    cpl_pqt=f'{INPUT_DIR}/cpl_{BASIN}_GLDAS.parquet', 
    Qex_ncf=QEXT_FILE
    )

Processing LSM data: 100%|██████████| 248/248 [00:00<00:00, 638.51it/s]


## 5. Create Cold Start File with `zeroqinit`

At the start of our simulation, we need to provide RAPID2 with initial values of discharge. We'll use the next command to create a cold start file with all zeroes.

In [9]:
QINIT_FILE = INPUT_DIR / f"Qinit_{BASIN}_GLDAS_2.1_VIC_2010-01.nc4"
r2.prep.coldinit(
    Qex_ncf=QEXT_FILE,
    Q00_ncf=QINIT_FILE
    )

## 6. Run the Routing Model with `rapid2`

RAPID2 uses a YAML configuration file (namelist) to specify input files, parameters, and output locations. `IS_dtR` is the routing time step (in seconds).

Before running the model, we need to generate the namelist file for our tutorial run. Here, we do this using the `yaml` python package to write the necessary input and output parameters to the namelist.

Run the model using the `rapid2` command.

In [10]:
r2.core.run(
    Qex_ncf=f'./{QEXT_FILE}',
    Q00_ncf=f'./{QINIT_FILE}',
    con_pqt=f'./{INPUT_DIR}/con_{BASIN}.parquet',
    kpr_pqt=f'./{INPUT_DIR}/kpr_{BASIN}_nrm.parquet',
    xpr_pqt=f'./{INPUT_DIR}/xpr_{BASIN}_nrm.parquet',
    bas_pqt=f'./{INPUT_DIR}/bas_{BASIN}_topo.parquet',
    IS_dtR=900,
    Qou_ncf=f'./{OUTPUT_DIR}/Qout_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4',
    Qfi_ncf=f'./{OUTPUT_DIR}/Qfinal_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4',
)

Computing discharge: 100%|██████████| 248/248 [00:05<00:00, 49.56it/s]


## 7. Plot Output
Now we can examine the output dishcarge. In this tutorial, we use the `netCDF4` python package to read our modeled output discharge file. This file contains time series of discharge for our selected month with time step set by the `IS_dtR` parameter in the namelist file. Here, we compute the mean discharge of every river reach in the basin, and then plot the discharge of reach with the highest mean.

In [30]:
with netCDF4.Dataset(f'./{OUTPUT_DIR}/Qout_{BASIN}_GLDAS_{PHASE}_{MODEL}_{TIME}_tst.nc4','r') as ds:
    qout = ds.variables["Qout"][:]
    rivid = ds.variables["rivid"][:]
    time = ds.variables['time'][:]
    date = []
    #print(ds.variables)
    for t in time:
        date.append(datetime.fromtimestamp(t))

    mean_per_reach = np.mean(qout, axis=0)
    main = int(np.argmax(mean_per_reach))
    peak_q=float(np.max(qout[:, main]))

    canvas = toyplot.Canvas(width=900,height=400,)
    axes = canvas.cartesian(label=f'Outflow for {BASIN} Time:{TIME}',xlabel='Date',ylabel=r'Discharge $[m^3/s]$')
    mark = axes.plot(time,qout[:,main])
    
    axes.x.ticks.show = True
    axes.x.ticks.locator = toyplot.locator.Timestamp()

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="900.0px" height="400.0px" viewBox="0 0 900.0 400.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t766165922ce04fdfb5f908af4381f453"> <path d="M 50.0 349.93982728719715 L 53.23886639676113 349.6688365221023 L 56.477732793522264 349.24778122901915 L 59.7165991902834 348.6924524307251 L 62.95546558704453 348.05086479187014 L 66.19433198380567 347.3471248626709 L 69.4331983805668 346.5974426269531 L 72.67206477732793 345.8164566040039 L 75.91093117408907 345.00624046325686 L 79.1497975708502 344.14132423400883 L 82.38866396761134 343.1750312805176 L 85.62753036437246 342.0632293701172 L 88.8663967611336 340.78268814086914 L 92.10526315789474 339.3342010498046 L 95.34412955465586 337.73556060791014 L 98.58299595141699 336.01203155517584 L 101.82186234817814 334.1889083862305 L 105.06072874493927 332.2877212524414 L 108.2995951417004 330.3256057739258 L 111.53846153846155 328.31634826660155 L 114.77732793522267 326.2718063354492 L 118.0161943319838 324.2031585693359 L 121.25506072874494 322.1218963623047 L 124.49392712550608 320.04051818847654 L 127.7327935222672 317.9728179931641 L 130.97165991902835 315.93358154296874 L 134.21052631578948 313.93775024414066 L 137.4493927125506 311.99925537109374 L 140.68825910931173 310.1298370361328 L 143.92712550607288 308.3382141113281 L 147.16599190283398 306.6297088623047 L 150.40485829959513 305.00640258789065 L 153.6437246963563 303.46766662597656 L 156.8825910931174 302.0108917236328 L 160.12145748987854 300.632177734375 L 163.36032388663966 299.3267028808594 L 166.5991902834008 298.08897094726564 L 169.83805668016194 296.91295166015624 L 173.0769230769231 295.79210205078124 L 176.31578947368422 294.71936645507816 L 179.55465587044534 293.68723144531253 L 182.79352226720647 292.68789062499997 L 186.0323886639676 291.7134643554687 L 189.27125506072875 290.75629272460935 L 192.51012145748987 289.8092468261719 L 195.748987854251 288.86602783203125 L 198.98785425101215 287.921337890625 L 202.22672064777328 286.97108764648436 L 205.4655870445344 286.0124145507812 L 208.70445344129558 285.0437316894532 L 211.94331983805668 284.06463623046875 L 215.1821862348178 283.0757751464844 L 218.42105263157896 282.0785888671875 L 221.65991902834008 281.0750366210937 L 224.8987854251012 280.06728515625 L 228.13765182186233 279.05739746093747 L 231.3765182186235 278.0470703125 L 234.6153846153846 277.0374572753907 L 237.85425101214577 276.0290466308594 L 241.0931174089069 275.02166748046875 L 244.33198380566802 274.0144897460937 L 247.57085020242917 273.0061706542969 L 250.80971659919027 271.99494018554685 L 254.04858299595142 270.978759765625 L 257.2874493927125 269.9554443359375 L 260.5263157894737 268.92279052734375 L 263.7651821862348 267.8786926269531 L 267.00404858299595 266.8212036132812 L 270.2429149797571 265.74863891601564 L 273.48178137651826 264.659619140625 L 276.7206477732793 263.5531066894531 L 279.9595141700405 262.428466796875 L 283.1983805668016 261.2854675292969 L 286.43724696356276 260.1242736816406 L 289.6761133603239 258.94544067382816 L 292.914979757085 257.74988403320316 L 296.1538461538462 256.53880615234374 L 299.3927125506073 255.31369018554688 L 302.63157894736844 254.07623291015625 L 305.87044534412956 252.82829589843752 L 309.1093117408907 251.571923828125 L 312.34817813765176 250.30930786132814 L 315.587044534413 249.0428283691406 L 318.8259109311741 247.7750549316406 L 322.0647773279352 246.50877685546874 L 325.3036437246963 245.24700927734375 L 328.5425101214575 243.99295654296873 L 331.7813765182186 242.75002441406247 L 335.02024291497975 241.52169189453124 L 338.25910